# MetaTrader 5 - Second-Based Candlestick Chart

This notebook resamples **MetaTrader 5 (MT5)** tick data into **second-based OHLC (Open, High, Low, Close) candlesticks**.

### Features:
1. Initialize connection to MT5 terminal.
2. Query tick range for a specific day (default: yesterday).
3. Preprocess tick data into a pandas DataFrame.
4. Resample tick data into second-based OHLC candlesticks with **editable timeframe** (`CANDLE_TIMEFRAME = '1s'`, `'5s'`, `'10s'`).
5. Visualize interactive Plotly Candlestick chart with synchronized tick volume bar chart and timeline range slider.


In [8]:
import MetaTrader5 as mt5
import pandas as pd
import numpy as np
from datetime import datetime, timedelta, UTC
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Set pandas options for better display
pd.set_option('display.max_columns', 10)
pd.set_option('display.width', 1000)

In [9]:
# Initialize MetaTrader 5 connection
if not mt5.initialize():
    print("MetaTrader5 initialization failed, error code:", mt5.last_error())
    quit()
else:
    print("MetaTrader5 initialized successfully.")

    # Print terminal connection info
    info = mt5.terminal_info()
    print(f"Connected to: {info.company} - {info.name}")
    print(f"Version: {mt5.version()}")

MetaTrader5 initialized successfully.
Connected to: RoboForex Ltd - RoboForex MT5 Terminal
Version: (500, 6090, '31 Jul 2026')


## Define Parameters

By default, we fetch ticks for the `EURUSD` symbol for **yesterday** (the day prior to current execution).
You can customize the `SYMBOL` and date range below.

In [10]:
# --- Configuration ---
SYMBOL = "USDJPY"  # Symbol to fetch
today = datetime.now()
yesterday = today - timedelta(days=1)

# Start and end of yesterday
date_from = datetime(yesterday.year, yesterday.month, yesterday.day, 0, 0, 0, tzinfo=UTC)
date_to = datetime(yesterday.year, yesterday.month, yesterday.day, 23, 59, 59, tzinfo=UTC)

print(f"Symbol: {SYMBOL}")
print(f"Date From: {date_from}")
print(f"Date To:   {date_to}")

Symbol: USDJPY
Date From: 2026-08-12 00:00:00+00:00
Date To:   2026-08-12 23:59:59+00:00


## Fetch Tick Data

We use `mt5.copy_ticks_range` to request tick data. The flag `mt5.COPY_TICKS_ALL` retrieves all tick types (bid, ask, etc.).

In [11]:
print(f"Requesting tick data for {SYMBOL}...")
ticks = mt5.copy_ticks_range(SYMBOL, date_from, date_to, mt5.COPY_TICKS_ALL)

if ticks is None or len(ticks) == 0:
    print(f"No ticks retrieved. Check if {SYMBOL} is available in Market Watch or if the market was open during the selected range.")
    print("Error code:", mt5.last_error())
else:
    print(f"Successfully retrieved {len(ticks):,} ticks.")

Requesting tick data for USDJPY...
Successfully retrieved 75,499 ticks.


## Preprocess Data

Let's convert the fetched ticks array into a pandas DataFrame, format the timestamps, and compute the raw spread (`ask - bid`).

To make the calculation universal, we fetch the symbol's information using `mt5.symbol_info` to get its `point` size and `digits` count. A standard **pip** is typically defined as 10 points for currency pairs with 3 or 5 decimal digits, and as 1 point for other symbols (such as gold, crypto, or indices).

In [12]:
if ticks is not None and len(ticks) > 0:
    # Create DataFrame
    df = pd.DataFrame(ticks)

    # Convert millisecond timestamp to pandas datetime
    df['time'] = pd.to_datetime(df['time_msc'], unit='ms')

    # Set the time column as the index for easier analysis
    df.set_index('time', inplace=True)

    # Fetch instrument info for universal pip calculation
    info = mt5.symbol_info(SYMBOL)
    if info is not None:
        point = info.point
        # A standard pip is 10 points for 3/5 digit forex pairs, and 1 point for others
        if info.digits in [3, 5]:
            pip_size = 10 * point
        else:
            pip_size = point
        print(f"Universal Pip Calculation: 1 Pip = {pip_size} (Point: {point}, Digits: {info.digits})")
    else:
        pip_size = 0.0001
        print(f"Symbol info not found for {SYMBOL}. Using fallback 1 Pip = {pip_size}")

    # Calculate bid-ask spread
    df['spread_raw'] = df['ask'] - df['bid']
    df['spread_pips'] = df['spread_raw'] / pip_size

    # Display the first few rows
    print("\nPreprocessed Tick DataFrame:")
    display(df.head())
else:
    print("No data to preprocess.")

Universal Pip Calculation: 1 Pip = 0.01 (Point: 0.001, Digits: 3)

Preprocessed Tick DataFrame:


,bid,ask,last,volume,time_msc,flags,volume_real,spread_raw,spread_pips
time,,,,,,,,,
2026-08-12 00:05:02.221,159.224,159.389,0.0,0,1786493102221,134,0.0,0.165,16.5
2026-08-12 00:05:02.285,159.222,159.389,0.0,0,1786493102285,130,0.0,0.167,16.7
2026-08-12 00:05:02.381,159.217,159.389,0.0,0,1786493102381,130,0.0,0.172,17.2
2026-08-12 00:06:49.036,159.224,159.358,0.0,0,1786493209036,134,0.0,0.134,13.4
2026-08-12 00:06:49.132,159.214,159.331,0.0,0,1786493209132,134,0.0,0.117,11.7


## Summary Statistics

Let's look at the basic statistics of the ticks, such as average, maximum, and minimum spreads.

In [13]:
if ticks is not None and len(ticks) > 0:
    print("--- Summary Statistics ---")
    print(f"Total Ticks: {len(df):,}")
    print(f"Min Bid: {df['bid'].min():.5f}")
    print(f"Max Ask: {df['ask'].max():.5f}")
    print(f"Average Spread (pips): {df['spread_pips'].mean():.2f}")
    print(f"Max Spread (pips): {df['spread_pips'].max():.2f}")
    print(f"Min Spread (pips): {df['spread_pips'].min():.2f}")
else:
    print("No data available for statistics.")

--- Summary Statistics ---
Total Ticks: 75,499
Min Bid: 158.57400
Max Ask: 159.54600
Average Spread (pips): 0.39
Max Spread (pips): 17.20
Min Spread (pips): 0.00


## Second-Based Candlestick Analysis

We resample tick data into **second-based OHLC (Open, High, Low, Close)** candlesticks. You can edit the `CANDLE_TIMEFRAME` parameter below to change the candle resolution (e.g., `'1s'`, `'5s'`, `'10s'`, `'30s'`, `'1min'`).

In [14]:
# --- Candle Timeframe Configuration ---
CANDLE_TIMEFRAME = "1s"  # Resample period: '1s' (1 sec), '5s', '10s', '30s', '1min', etc.

if ticks is not None and len(ticks) > 0:
    print(f"Resampling tick data into {CANDLE_TIMEFRAME} candlestick bars...")

    # Resample bid prices into OHLC (Open, High, Low, Close)
    df_candles = df['bid'].resample(CANDLE_TIMEFRAME).ohlc()

    # Add tick volume (number of ticks per interval)
    df_candles['tick_volume'] = df['bid'].resample(CANDLE_TIMEFRAME).count()

    # Add average spread (in pips) for the candle
    df_candles['avg_spread_pips'] = df['spread_pips'].resample(CANDLE_TIMEFRAME).mean()

    # Drop intervals with no tick activity
    df_candles.dropna(subset=['open'], inplace=True)

    print(f"Successfully generated {len(df_candles):,} candlestick bars ({CANDLE_TIMEFRAME}).")
    display(df_candles.head())

    # Create Subplot Figure: Top = Candlestick Chart, Bottom = Tick Volume
    fig_candles = make_subplots(
        rows=2, cols=1,
        shared_xaxes=True,
        vertical_spacing=0.08,
        row_heights=[0.7, 0.3],
        subplot_titles=(f"{SYMBOL} {CANDLE_TIMEFRAME} Candlestick Chart (Bid)", "Tick Volume (Ticks per Bar)")
    )

    # 1. Candlestick Trace
    fig_candles.add_trace(
        go.Candlestick(
            x=df_candles.index,
            open=df_candles['open'],
            high=df_candles['high'],
            low=df_candles['low'],
            close=df_candles['close'],
            name='OHLC',
            increasing_line_color='#26a69a',
            decreasing_line_color='#ef5350'
        ),
        row=1, col=1
    )

    # 2. Tick Volume Bar Trace
    fig_candles.add_trace(
        go.Bar(
            x=df_candles.index,
            y=df_candles['tick_volume'],
            name='Tick Volume',
            marker_color='#787c99',
            opacity=0.7,
            hovertemplate="<b>Time:</b> %{x|%Y-%m-%d %H:%M:%S}<br><b>Volume:</b> %{y} ticks<extra>Volume</extra>"
        ),
        row=2, col=1
    )

    # Layout & Range Slider Setup
    date_str = df_candles.index[0].strftime('%Y-%m-%d') if len(df_candles) > 0 else ""
    fig_candles.update_layout(
        title=dict(
            text=f"MetaTrader 5 - {SYMBOL} {CANDLE_TIMEFRAME} Interactive Candlestick Analysis ({date_str})",
            x=0.5,
            xanchor='center'
        ),
        height=750,
        hovermode='x unified',
        template='plotly_white',
        xaxis_rangeslider_visible=False,  # Disable upper slider to keep timeline clear
        showlegend=True,
        legend=dict(
            orientation='h',
            yanchor='bottom',
            y=1.02,
            xanchor='right',
            x=1
        ),
        margin=dict(l=60, r=40, t=100, b=60)
    )

    # Configure Bottom Subplot Range Slider
    fig_candles.update_xaxes(
        row=2, col=1,
        rangeslider=dict(visible=True, thickness=0.08),
        type='date'
    )

    fig_candles.update_yaxes(title_text="Price (Bid)", row=1, col=1)
    fig_candles.update_yaxes(title_text="Tick Volume", row=2, col=1)

    fig_candles.show()
else:
    print("No tick data available to generate candlesticks.")

Resampling tick data into 1s candlestick bars...
Successfully generated 31,588 candlestick bars (1s).


,open,high,low,close,tick_volume,avg_spread_pips
time,,,,,,
2026-08-12 00:05:02,159.224,159.224,159.217,159.217,3,16.800000
2026-08-12 00:06:49,159.224,159.224,159.214,159.219,6,11.633333
2026-08-12 00:06:50,159.219,159.220,159.219,159.220,2,11.350000
2026-08-12 00:06:51,159.221,159.221,159.220,159.221,3,11.133333
2026-08-12 00:06:52,159.220,159.221,159.220,159.220,6,11.133333
